# Meta-Topic Clustering Test (YouTube)

This notebook creates a **second layer** of `MetaTopic` nodes, starting from topic arrays stored in MongoDB `youtube_channel_videos`.

What it does:
- Reads per-video topic arrays from MongoDB (`comments_frequent_topics` and related fields)
- Seeds local Neo4j with `(:YouTubeVideo)-[:HAS_COMMENT_TOPIC]->(:Topic)`
- Builds embeddings for topic names (OpenAI or Azure OpenAI)
- Clusters similar topics into meta-topics
- Writes `(:Topic)-[:IN_META_TOPIC]->(:MetaTopic)` back to Neo4j
- Runs validation queries

Use this as a test harness before productionizing in Airflow.

In [1]:
import sys, os
print("python:", sys.executable)
print("cwd:", os.getcwd())

python: /home/airflow/.local/bin/python
cwd: /opt/airflow/notebooks


In [18]:
import os
from datetime import datetime

import numpy as np  # pyright: ignore[reportMissingImports]
import pandas as pd  # pyright: ignore[reportMissingImports]
from neo4j import GraphDatabase  # pyright: ignore[reportMissingImports]
from pymongo import MongoClient  # pyright: ignore[reportMissingImports]
from sklearn.cluster import AgglomerativeClustering  # pyright: ignore[reportMissingImports]
from sklearn.metrics.pairwise import cosine_distances  # pyright: ignore[reportMissingImports]  # pyright: ignore[reportMissingImports]
from openai import AzureOpenAI  # pyright: ignore[reportMissingImports]

try:
    from dotenv import load_dotenv  # pyright: ignore[reportMissingImports]
    load_dotenv(".env")
except Exception:
    pass

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

# ---- Config ----
# Mongo
MONGODB_URI = os.getenv("MONGODB_URI")
MONGODB_DB = os.getenv("MONGODB_DB", "rbl")
MONGO_VIDEOS_COLLECTION = os.getenv("MONGO_VIDEOS_COLLECTION", "youtube_channel_videos")
MONGO_TIMEOUT_MS = int(os.getenv("META_TOPIC_MONGO_TIMEOUT_MS", "15000"))

# Local Neo4j
NEO4J_URI_DEV = os.getenv("NEO4J_URI_DEV", "bolt://localhost:7687")
NEO4J_USER_DEV = os.getenv("NEO4J_USER_DEV", "neo4j")
NEO4J_PASSWORD_DEV = os.getenv("NEO4J_PASSWORD_DEV")

# Azure OpenAI embeddings
EMBED_BATCH_SIZE = int(os.getenv("META_TOPIC_EMBED_BATCH_SIZE", "100"))
EMBED_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")

# Clustering threshold (cosine distance). Lower = stricter/smaller clusters.
CLUSTER_DISTANCE_THRESHOLD = float(os.getenv("META_TOPIC_DISTANCE_THRESHOLD", "0.28"))

# Namespace to make reruns idempotent/versioned
CLUSTER_VERSION = os.getenv("META_TOPIC_CLUSTER_VERSION", datetime.utcnow().strftime("%Y%m%d_%H%M%S"))

if not MONGODB_URI:
    raise ValueError("Set MONGODB_URI in environment before running this notebook.")
if not NEO4J_PASSWORD_DEV:
    raise ValueError("Set NEO4J_PASSWORD_DEV in environment before running this notebook.")
if not EMBED_MODEL:
    raise ValueError("Set AZURE_OPENAI_EMBEDDING_MODEL in environment before running this notebook.")

print("Mongo DB:", MONGODB_DB)
print("Mongo collection:", MONGO_VIDEOS_COLLECTION)
print("NEO4J_URI_DEV:", NEO4J_URI_DEV)
print("EMBED_MODEL:", EMBED_MODEL)
print("CLUSTER_DISTANCE_THRESHOLD:", CLUSTER_DISTANCE_THRESHOLD)
print("CLUSTER_VERSION:", CLUSTER_VERSION)

Mongo DB: rbl
Mongo collection: youtube_channel_videos
NEO4J_URI_DEV: bolt://neo4j:7687
EMBED_MODEL: text-embedding-3-small
CLUSTER_DISTANCE_THRESHOLD: 0.28
CLUSTER_VERSION: 20260311_184624


/tmp/ipykernel_1226/2820081638.py:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  CLUSTER_VERSION = os.getenv("META_TOPIC_CLUSTER_VERSION", datetime.utcnow().strftime("%Y%m%d_%H%M%S"))


In [3]:
# Mongo ping
from pymongo import MongoClient  # pyright: ignore[reportMissingImports]
mc = MongoClient(MONGODB_URI)
print(mc.admin.command("ping"))

/tmp/ipykernel_1226/2845409533.py:3: UserWarning: You appear to be connected to a CosmosDB cluster. For more information regarding feature compatibility and support please visit https://www.mongodb.com/supportability/cosmosdb
  mc = MongoClient(MONGODB_URI)


{'ok': 1.0}


In [4]:
# Neo4j ping
from neo4j import GraphDatabase  # pyright: ignore[reportMissingImports]
driver = GraphDatabase.driver(NEO4J_URI_DEV, auth=(NEO4J_USER_DEV, NEO4J_PASSWORD_DEV))
with driver.session() as s:
    print(s.run("RETURN 1 AS ok").single()["ok"])

1


In [19]:
def get_openai_client():
    azure_endpoint = (os.getenv("AZURE_OPENAI_ENDPOINT") or "").strip().rstrip("/")
    azure_api_key = os.getenv("AZURE_OPENAI_API_KEY")
    azure_api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")

    if not azure_endpoint or not azure_api_key:
        raise ValueError("Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY.")

    print("Using Azure OpenAI embeddings")
    return AzureOpenAI(
        api_key=azure_api_key,
        api_version=azure_api_version,
        azure_endpoint=azure_endpoint,
    )


def batched(items, size):
    for i in range(0, len(items), size):
        yield items[i : i + size]


def fetch_video_topic_rows_from_mongo(mongo_uri, mongo_db, collection_name):
    client = MongoClient(mongo_uri)
    coll = client[mongo_db][collection_name]

    query = {
        "video_id": {"$exists": True, "$nin": [None, ""]},
        "comments_frequent_topics": {"$exists": True, "$type": "array"},
    }
    projection = {
        "_id": 0,
        "video_id": 1,
        "comments_frequent_topics": 1,
        "comments_frequent_topic_categories": 1,
        "comments_frequent_topic_weights": 1,
    }

    rows = []
    for doc in coll.find(query, projection=projection):
        video_id = str(doc.get("video_id", "")).strip()
        if not video_id:
            continue

        topics = doc.get("comments_frequent_topics") or []
        categories = doc.get("comments_frequent_topic_categories") or []
        weights = doc.get("comments_frequent_topic_weights") or []

        if not isinstance(topics, list):
            continue

        for idx, topic in enumerate(topics):
            topic_name = str(topic or "").strip()
            if not topic_name:
                continue

            category = ""
            if isinstance(categories, list) and idx < len(categories):
                category = str(categories[idx] or "").strip()

            weight = 0.0
            if isinstance(weights, list) and idx < len(weights):
                try:
                    weight = float(weights[idx] or 0.0)
                except Exception:
                    weight = 0.0

            rows.append(
                {
                    "video_id": video_id,
                    "topic_name": topic_name,
                    "topic_category": category,
                    "weight": weight,
                }
            )

    client.close()
    return pd.DataFrame(rows)


def aggregate_topic_stats(video_topic_df):
    if video_topic_df.empty:
        return pd.DataFrame(columns=["topic_name", "videos_count", "total_weight"])

    agg = (
        video_topic_df.groupby("topic_name", as_index=False)
        .agg(
            videos_count=("video_id", "nunique"),
            total_weight=("weight", "sum"),
        )
        .sort_values("total_weight", ascending=False)
        .reset_index(drop=True)
    )
    return agg


def seed_local_neo4j_with_topics(driver, video_topic_df, batch_size=1000):
    if video_topic_df.empty:
        return 0

    rows = video_topic_df.to_dict("records")
    now_iso = datetime.utcnow().isoformat()

    write_q = """
    UNWIND $rows AS row
    MERGE (v:YouTubeVideo {video_id: row.video_id})
    MERGE (t:Topic {name: row.topic_name})
    SET t.category = CASE
      WHEN row.topic_category IS NULL OR row.topic_category = '' THEN t.category
      ELSE row.topic_category
    END
    MERGE (v)-[r:HAS_COMMENT_TOPIC]->(t)
    SET r.weight = coalesce(row.weight, 0.0),
        r.source = 'youtube_channel_videos',
        r.updated_at = datetime($now_iso)
    """

    with driver.session() as session:
        for chunk in batched(rows, batch_size):
            session.run(write_q, rows=chunk, now_iso=now_iso)

    return len(rows)


def get_embeddings(client, texts, model, batch_size=100):
    vectors = []
    for chunk in batched(texts, batch_size):
        resp = client.embeddings.create(model=model, input=chunk)
        vectors.extend([d.embedding for d in resp.data])
    return np.array(vectors, dtype=np.float32)

In [6]:
driver = GraphDatabase.driver(
    NEO4J_URI_DEV,
    auth=(NEO4J_USER_DEV, NEO4J_PASSWORD_DEV),
)

client = get_openai_client()

video_topic_df = fetch_video_topic_rows_from_mongo(
    mongo_uri=MONGODB_URI,
    mongo_db=MONGODB_DB,
    collection_name=MONGO_VIDEOS_COLLECTION,
)
if video_topic_df.empty:
    raise RuntimeError("No topic rows found in Mongo youtube_channel_videos fields.")

seeded_rows = seed_local_neo4j_with_topics(driver, video_topic_df)
print(f"Seeded local Neo4j with {seeded_rows} video-topic rows")

df = aggregate_topic_stats(video_topic_df)
if df.empty:
    raise RuntimeError("No aggregated topics could be built from Mongo rows.")

df = df.dropna(subset=["topic_name"]).copy()
df["topic_name"] = df["topic_name"].astype(str).str.strip()
df = df[df["topic_name"] != ""].reset_index(drop=True)

print(f"Fetched {len(df)} unique topics from Mongo")
df.head(10)

Using Azure OpenAI embeddings


/tmp/ipykernel_1226/2597545332.py:28: UserWarning: You appear to be connected to a CosmosDB cluster. For more information regarding feature compatibility and support please visit https://www.mongodb.com/supportability/cosmosdb
  client = MongoClient(mongo_uri)
/tmp/ipykernel_1226/2597545332.py:106: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now_iso = datetime.utcnow().isoformat()
[#C0E8]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('neo4j', 7687)) (ResolvedIPv4Address(('172.18.0.6', 7687))): OSError('No data')


ServiceUnavailable: Failed to read from defunct connection IPv4Address(('neo4j', 7687)) (ResolvedIPv4Address(('172.18.0.6', 7687)))

In [8]:
# Fallback: build df directly from Neo4j (if Mongo->seed cell fails)
query = """
MATCH (v:YouTubeVideo)-[r:HAS_COMMENT_TOPIC]->(t:Topic)
WITH
  t.name AS topic_name,
  count(DISTINCT v) AS videos_count,
  sum(coalesce(r.weight, 0.0)) AS total_weight
RETURN topic_name, videos_count, total_weight
ORDER BY total_weight DESC
"""

with driver.session() as session:
    rel_count = session.run(
        "MATCH (:YouTubeVideo)-[r:HAS_COMMENT_TOPIC]->(:Topic) RETURN count(r) AS c"
    ).single()["c"]
    print("HAS_COMMENT_TOPIC rels:", rel_count)

    records = session.run(query).data()

df = pd.DataFrame(records)
if df.empty:
    raise RuntimeError("No Topic data found in Neo4j for HAS_COMMENT_TOPIC.")

df = df.dropna(subset=["topic_name"]).copy()
df["topic_name"] = df["topic_name"].astype(str).str.strip()
df = df[df["topic_name"] != ""].reset_index(drop=True)

print(f"Loaded {len(df)} unique topics from Neo4j")
display(df.head(10))

HAS_COMMENT_TOPIC rels: 71645
Loaded 57558 unique topics from Neo4j


,topic_name,videos_count,total_weight
0,Humor and Laughter,192,60.860
1,Requests for More Content,173,27.630
2,Fortnite Gameplay,99,24.580
3,Humor and Memes,152,23.760
4,Humor and Entertainment,76,20.030
5,Laughter and Humor,48,19.330
6,Positive Feedback,103,17.795
7,Among Us Mods,28,16.510
8,Video Appreciation,82,15.570
9,Humor and Funny Moments,68,15.330


In [20]:
# Guard for out-of-order execution in notebooks.
if "client" not in globals() or client is None:
    client = get_openai_client()

texts = df["topic_name"].tolist()
emb = get_embeddings(client, texts, model=EMBED_MODEL, batch_size=EMBED_BATCH_SIZE)

# Agglomerative clustering using cosine distance matrix.
dist = cosine_distances(emb)

clusterer = AgglomerativeClustering(
    metric="precomputed",
    linkage="average",
    distance_threshold=CLUSTER_DISTANCE_THRESHOLD,
    n_clusters=None,
)
labels = clusterer.fit_predict(dist)

df["cluster_id"] = labels

# Build deterministic meta-topic IDs per cluster for this run version.
cluster_to_meta = {c: f"yt_meta_{CLUSTER_VERSION}_{c}" for c in sorted(df["cluster_id"].unique())}
df["meta_topic_id"] = df["cluster_id"].map(cluster_to_meta)

# Pick a cluster label: highest total_weight topic in each cluster.
cluster_labels = {}
for c, g in df.groupby("cluster_id"):
    top_row = g.sort_values(["total_weight", "videos_count"], ascending=False).iloc[0]
    cluster_labels[c] = str(top_row["topic_name"])

df["meta_topic_label"] = df["cluster_id"].map(cluster_labels)

print("Clusters:", df["cluster_id"].nunique())
(
    df.groupby(["cluster_id", "meta_topic_label"], as_index=False)
      .agg(topic_count=("topic_name", "count"),
           videos_count=("videos_count", "sum"),
           total_weight=("total_weight", "sum"))
      .sort_values("total_weight", ascending=False)
      .head(30)
)

NotFoundError: Error code: 404 - {'error': {'code': 'DeploymentNotFound', 'message': 'The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.'}}

In [ ]:
# Persist to Neo4j
# Creates:
#   (:MetaTopic {meta_topic_id, label, version, source, created_at})
#   (:Topic)-[:IN_META_TOPIC {version, score}]->(:MetaTopic)

rows = df.to_dict("records")
created_at = datetime.utcnow().isoformat()

write_q = """
UNWIND $rows AS row
MERGE (t:Topic {name: row.topic_name})
MERGE (m:MetaTopic {meta_topic_id: row.meta_topic_id})
SET m.label = row.meta_topic_label,
    m.version = $version,
    m.source = 'youtube_meta_topic_notebook',
    m.created_at = datetime($created_at)
MERGE (t)-[r:IN_META_TOPIC {version: $version}]->(m)
SET r.score = coalesce(row.total_weight, 0.0)
"""

with driver.session() as session:
    session.run(write_q, rows=rows, version=CLUSTER_VERSION, created_at=created_at)

print(f"Wrote {len(rows)} Topic -> MetaTopic mappings for version={CLUSTER_VERSION}")

In [ ]:
# Validation queries

q1 = """
MATCH (m:MetaTopic {version: $version})<-[:IN_META_TOPIC {version: $version}]-(t:Topic)
RETURN m.meta_topic_id AS meta_topic_id,
       m.label AS label,
       count(t) AS topic_count
ORDER BY topic_count DESC
LIMIT 20
"""

q2 = """
MATCH (v:YouTubeVideo)-[h:HAS_COMMENT_TOPIC]->(t:Topic)-[:IN_META_TOPIC {version: $version}]->(m:MetaTopic)
RETURN m.label AS meta_topic,
       count(DISTINCT v) AS videos,
       sum(coalesce(h.weight, 0.0)) AS weighted_mentions
ORDER BY weighted_mentions DESC
LIMIT 20
"""

with driver.session() as session:
    top_clusters = [dict(r) for r in session.run(q1, version=CLUSTER_VERSION)]
    top_meta = [dict(r) for r in session.run(q2, version=CLUSTER_VERSION)]

print("Top clusters by topic count")
display(pd.DataFrame(top_clusters))
print("\nTop meta-topics by weighted mentions")
display(pd.DataFrame(top_meta))

## Next tests

Try these tuning options:
- `META_TOPIC_DISTANCE_THRESHOLD=0.24` for tighter clusters
- `META_TOPIC_DISTANCE_THRESHOLD=0.32` for broader clusters
- rerun with a new `META_TOPIC_CLUSTER_VERSION` and compare outputs

Useful Cypher to fetch comments under a meta-topic:

```cypher
MATCH (c:YouTubeComment)-[:BELONGS_TO]->(v:YouTubeVideo)-[:HAS_COMMENT_TOPIC]->(t:Topic)-[:IN_META_TOPIC {version: $version}]->(m:MetaTopic)
WHERE m.label = $meta_topic_label
RETURN c.comment_id, c.textOriginal, v.video_id
LIMIT 100;
```

If your graph does not have `(:YouTubeComment)-[:BELONGS_TO]->(:YouTubeVideo)`, adapt the path to your existing comment-video relationship.